# SIH1707 — Geolocation-Based Attendance Tracking App
**Org:** GAIL (India) Ltd, Ministry of Petroleum and Natural Gas | **Category:** Software · Miscellaneous

**Constraint:** No hardware/software/licenses/data provided by GAIL — free/open-source only.

## 1. Problem Restated
1. **Auto check-in/out** when entering/leaving a 200m radius of an office.
2. **Every check-in pairs with a check-out**, no matter how many times the employee enters/exits in a day.
3. **Manual check-in/out for offsite work**, with the app suggesting nearby known locations from live lat/long.
4. **Total working hours** computed per employee per day.
5. **Tamper-proof, accurate records** with real-time sync and no data loss.

## 2. What Actually Wins This
Basic check-in/out on radius entry/exit is table stakes. The competition is won on three things the spec implies but doesn't spell out:

- **Anti-spoofing** — GAIL cannot deploy an attendance system that's trivially fakeable with a mock-location app. Single biggest differentiator.
- **The pairing invariant under messy real-world GPS** — boundary jitter, dead phones, force-kills, inter-office jumps all break a naive implementation.
- **Reliable background operation** — Android Doze/OEM battery killers and iOS background suspension are the actual reason most geofencing apps fail in the field, not the geofencing logic itself.

Build the happy path fast, spend the majority of remaining time on these three. **Demo should actively try to spoof the app on stage and show it getting caught.**

## 3. Tech Stack (free/open-source only)

| Layer | Choice | Why |
|---|---|---|
| Mobile frontend | React Native (Expo, bare workflow if needed) | Fast to scaffold, cross-platform, large ecosystem |
| Geofencing | `react-native-background-geolocation` (free tier) or native `Geofencing API` (Android) + `CLCircularRegion` (iOS) | OS-level geofencing, not polling — critical for battery + reliability |
| Maps / geocoding | OpenStreetMap + Nominatim (reverse geocode) + Overpass API (nearby place suggestions) | GAIL forbids paid APIs — no Google Maps Platform |
| Backend | FastAPI (Python) | Team familiarity |
| Database | PostgreSQL (+ PostGIS if time allows; plain Haversine fallback otherwise) | Native geospatial queries vs hand-rolled Haversine everywhere |
| Auth | JWT + refresh tokens | Standard, no external dependency |
| Local offline storage | SQLite (WatermelonDB or expo-sqlite) | Queue events when offline, sync later |
| Admin dashboard | React + Leaflet.js (OSM tiles) | Live map, anomaly queue, timesheets |
| Push/sync | WebSocket or polling + background sync task | Real-time sync requirement |

**DB note:** Postgres/PostGIS setup being handled by the team directly (not scaffolded by this plan).

## 4. Data Model (core tables)
```
employees
  id, name, employee_code, office_id (home office), role

offices
  id, name, lat, lng, radius_m (default 200)

attendance_events
  id, employee_id, office_id (nullable for offsite),
  event_type (check_in / check_out),
  source (auto_geofence / manual_offsite),
  lat, lng, accuracy_m, device_id,
  is_mock_flag, is_anomaly_flag,
  server_ts, client_ts, synced_at

sessions
  id, employee_id, check_in_event_id, check_out_event_id (nullable while open),
  status (open / closed / force_closed),
  total_minutes

offsite_locations
  id, name, lat, lng, added_by (for reuse/suggestions)
```
`sessions` is the reconciliation layer that enforces the pairing invariant — never trust raw events alone for hours calculation.

## 5. Core Logic Design

### 5.1 Geofence state machine (solves the pairing invariant)
Do not fire check-in/check-out directly off raw "inside radius" booleans — GPS jitter near a 200m boundary generates dozens of false transitions.
- **Hysteresis**: enter threshold 200m, exit threshold 220-250m.
- **Dwell time**: signal must persist inside/outside for 60-90s before committing a state transition.
- **End-of-day auto-close**: any open session with no check-out gets `force_closed`, flagged for HR review — never silently dropped.
- **No concurrent open sessions**: entering Office B while a session at Office A is open force-closes A and opens a new one at B.

### 5.2 Anti-spoofing (build this early, not last)
- **Mock location detection**: check Android's `Location.isFromMockProvider()` on every event; reject/flag if true.
- **Plausibility checks**: reject if implied speed since the last fix exceeds ~150 km/h — catches GPS teleportation.
- **Environmental cross-check** (stretch): visible WiFi BSSIDs / cell tower IDs alongside GPS — spoofing GPS is easy, spoofing a consistent radio environment isn't.
- **Device attestation** (stretch): Play Integrity API to detect rooted/emulated devices at login.
- **Accuracy filtering**: discard fixes with `accuracy_m` > 50m.
- All flagged events land in an **admin anomaly queue** — human-in-the-loop review, never silent auto-reject.

### 5.3 Offsite manual check-in
Reverse-geocode via Nominatim, suggest previously-used nearby points from `offsite_locations`, allow adding new. Same anomaly checks as auto events.

### 5.4 Working hours calculation
Sum `total_minutes` across all `closed` and `force_closed` sessions per employee per day. Force-closed sessions flagged in the timesheet UI (estimated vs. confirmed hours).

### 5.5 Offline-first sync
All events write to local SQLite immediately regardless of connectivity. Background sync pushes queued events when network returns; server dedupes on a client-generated event UUID. Local queue is source of truth on-device until server ack.

## 6. Team & Task Division (6 people, ranked Vansh > Ditya > Manya > Ankur > Arush > Parth)

| Person | Role | Tasks |
|---|---|---|
| **Vansh** | Mobile/Geofencing Lead | Hardest, most failure-prone piece. State machine: enter/exit hysteresis, dwell timer, force-close-on-office-switch, reliable background execution on real devices (Doze/battery whitelist) |
| **Ditya** | Backend Lead | Schema, event ingest with UUID dedupe, session reconciliation (pairing invariant, end-of-day auto-close), working-hours calc |
| **Manya** | Anti-Spoofing | Differentiator, semi-independent once event schema is fixed. Tier 1 mandatory: mock-flag, speed/plausibility check, accuracy filter. Tier 2 stretch: WiFi/cell cross-check |
| **Ankur** | Offsite flow + offline sync (mobile support, under Vansh) | Manual check-in UI, Nominatim reverse-geocode + suggestions, expo-sqlite local queue, background sync/retry |
| **Arush** | Admin dashboard | React + Leaflet live map, anomaly review queue, timesheet view + CSV export |
| **Parth** | Seed data, QA, demo support | Seed 2-3 mock GAIL offices, write spoof-and-catch + offline-resilience test scenarios, own rehearsal checklist/timing, help Arush with dashboard polish |

**Risk mitigation (§9):** everyone does a 10-minute walkthrough of every other member's piece before the final rehearsal — no unowned code.

## 7. Build Timeline (36-hour hackathon)

| Hours | Milestone |
|---|---|
| 0-4 | Repo, backend skeleton, mobile skeleton, auth (JWT), seed 2-3 mock GAIL offices |
| 4-10 | Core geofencing happy path: native geofence registration, basic enter→check-in / exit→check-out, events synced to backend, basic "today's status" screen |
| 10-16 | Pairing invariant + reconciliation: hysteresis + dwell logic, force-close-on-conflict, end-of-day auto-close, working hours calc |
| 16-24 | Anti-spoofing layer (priority differentiator): mock-flag capture, speed/plausibility check, accuracy filtering, anomaly flag surfaced in admin queue. Stretch if time: WiFi/cell cross-check, Play Integrity |
| 24-30 | Offsite manual flow + offline sync: Nominatim reverse-geocode + suggestions, SQLite local queue, background sync/dedupe, kill network mid-demo and confirm recovery |
| 30-34 | Admin dashboard: Leaflet live map, anomaly review queue, timesheet view + CSV export |
| 34-36 | Demo prep: script the spoof-and-catch moment, rehearse full flow twice on actual demo devices, one person owns the pitch narrative |

## 8. Demo Script (this is what wins)
1. **Happy path** (60s): walk into the geofence, show auto check-in fire live; walk out, show check-out; show the session in the dashboard with computed hours.
2. **The pairing stress test** (30s): re-enter/exit rapidly near the boundary on stage — show it does *not* generate spurious events, explain hysteresis/dwell in one sentence.
3. **The spoof attempt** (60s — the differentiator): turn on Android mock location, "teleport" the device to the office. Show the event land in the **anomaly queue flagged as mock**, not silently accepted.
4. **Offsite flow** (30s): manual check-in from a random location, show the suggested-location list from Nominatim/Overpass.
5. **Offline resilience** (30s): airplane mode on, check-in still records locally; airplane mode off, show it sync to the dashboard.
6. **Close on the constraint they actually care about**: "Built entirely on free and open-source infrastructure, per the problem statement — no Google Maps billing, no paid geocoding, deployable as-is."

## 9. Risks & Mitigations

| Risk | Mitigation |
|---|---|
| Background geofencing unreliable on demo devices (OEM battery killers) | Test on the *actual* demo phones early, not emulators. Whitelist the app from battery optimization before presenting |
| GPS accuracy indoors/at venue is poor | Widen radius for demo purposes if needed, be upfront about it — judges understand indoor GPS limitations |
| Running out of time on anti-spoofing | Scoped in 3 tiers (mock-flag → plausibility → radio cross-check); ship tier 1 no matter what, treat 2-3 as stretch |
| PostGIS/DB setup friction under time pressure | Fallback: plain lat/lng + Haversine formula in application code if PostGIS setup stalls |
| Team can't explain a component under judge questioning | Everyone does a 10-minute walkthrough of every other member's piece before the final rehearsal — no unowned code |

## 10. Stretch Goals (only after §7 core is done)
- WiFi/cell tower cross-verification for spoofing defence
- Leave/holiday calendar integration affecting expected hours
- Push notifications for missed check-out reminders
- Analytics: attendance trends, late-arrival patterns, per-office heatmap